# CYR-GPU-014 / R1C — CUDA preflight diagnostic

Use a **T4 GPU** and run the single cell below. This does **not** run scientific arms. It executes the already-frozen R1C preflight at commit `2a71cea10ebb7a231834b6b112c49e268e9631a5`, captures the child process stdout/stderr, and prints the exact failing stage. Existing Drive data is preserved.


In [ ]:
import json, subprocess, sys
from pathlib import Path
REPO=Path('/content/An-Ra-the-new-AGI-r1c-debug')
REMOTE='https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git'
BRANCH='cymek-500m-readiness'
EXEC='2a71cea10ebb7a231834b6b112c49e268e9631a5'
PRE_REL='docs/cymek/experiments/CYR-GPU-014-R1C/PREREGISTRATION.json'
if not REPO.exists(): subprocess.run(['git','clone','--branch',BRANCH,'--single-branch','--depth','160',REMOTE,str(REPO)],check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','origin',BRANCH,'--depth','160'],check=True)
    subprocess.run(['git','-C',str(REPO),'checkout','-q',BRANCH],check=True)
    subprocess.run(['git','-C',str(REPO),'reset','--hard',f'origin/{BRANCH}'],check=True)
pre_text=(REPO/PRE_REL).read_text(); PRE_LOCAL=Path('/content/CYR_GPU_014_R1C_PREREGISTRATION.json'); PRE_LOCAL.write_text(pre_text)
subprocess.run(['git','-C',str(REPO),'checkout','-q',EXEC],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','pytest','numpy'],check=True)
import torch
if not torch.cuda.is_available(): raise RuntimeError('Select Runtime -> Change runtime type -> T4 GPU')
print('GPU:',torch.cuda.get_device_name(0),'VRAM GiB:',round(torch.cuda.get_device_properties(0).total_memory/2**30,2))
from google.colab import drive
drive.mount('/content/drive')
OUT=Path('/content/drive/MyDrive/CYMEK/CYR-GPU-014-R1C'); OUT.mkdir(parents=True,exist_ok=True)
cmd=[sys.executable,'anra_v5/cyr_gpu014_r1c_run.py','--mode','preflight','--repo',str(REPO),'--out',str(OUT),'--prereg',str(PRE_LOCAL)]
print('RUNNING:', ' '.join(cmd), flush=True)
p=subprocess.run(cmd,cwd=REPO,text=True,capture_output=True)
print('\n===== CHILD STDOUT =====')
print(p.stdout or '<empty>')
print('===== CHILD STDERR =====')
print(p.stderr or '<empty>')
print('===== RETURN CODE =====',p.returncode)
print('\n===== PREFLIGHT ARTIFACT SCAN =====')
for name in ['EXACT_RESUME_SMOKE.json','PREEXECUTION_EXACT_RESUME_SMOKE.pt','CALIBRATION.json','RESOLVED.json','MATCHED_INIT_PREFLIGHT.json','PREEXECUTION_GATE.json']:
    q=OUT/name
    print(name, 'EXISTS' if q.exists() else 'MISSING', q.stat().st_size if q.exists() else '')
if p.returncode==0:
    print('\nDIAGNOSIS: PREFLIGHT PASSED. The original notebook can now reuse PREEXECUTION_GATE.json and proceed to Cell 1.')
else:
    stage='EXACT_RESUME_SMOKE_OR_EARLIER'
    if 'R1C calibration:' in p.stdout: stage='CALIBRATION_OR_LATER'
    print('\nDIAGNOSIS STAGE:',stage)
    print('Do not run scientific Cell 1 yet. Send ChatGPT the CHILD STDERR plus the last ~30 lines of CHILD STDOUT.')
